# FundusNet — Colab Training Pipeline

Train 5 models (Swin, MaxViT, ConvNeXt V2, EfficientNet V2, DeiT III) on GPU,
export to ONNX, and download for local deployment.

**Runtime:** GPU (T4) recommended. Go to `Runtime > Change runtime type > T4 GPU`.

## 1. Setup — Clone repo & install dependencies

In [ ]:
# Clone your repo (replace with your actual repo URL)
!git clone https://github.com/yourusername/FundusNet.git
%cd FundusNet

# Install dependencies
!pip install -r requirements.txt
!pip install timm onnxruntime

## 2. Upload Dataset

Upload your `retina_dataset/` folder. It should contain 4 subfolders:
- `Healthy/` (~1074 images)
- `Cataract/` (~1038 images)
- `Glaucoma/` (~1007 images)
- `Retina Disease/` (~1098 images)

In [ ]:
# Option A: Upload from local machine (runs a file picker)
import os
import zipfile

from google.colab import files

# Zip your retina_dataset folder locally first, then upload here
print("Upload your retina_dataset.zip:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall(".")
        print(f"Extracted {filename}")

# Verify dataset structure
dataset_path = "retina_dataset"
for cls in ["Healthy", "Cataract", "Glaucoma", "Retina Disease"]:
    cls_path = os.path.join(dataset_path, cls)
    if os.path.exists(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
        print(f"  {cls}: {count} images")
    else:
        print(f"  {cls}: MISSING!")

## 3. Verify GPU is available

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Training will be very slow.")
    print("Go to Runtime > Change runtime type > GPU")

## 4. Train All 5 Models

Each model trains with 5-fold cross-validation, 200 epochs.
Estimated time on T4 GPU: ~4-6 hours total.

In [ ]:
# Train all 5 models
!python train.py --models swin maxvit convnext_v2 efficientnet_v2 deit --epochs 200 --batch-size 32 --folds 5

## 5. Export to ONNX

Export each trained model to ONNX format for 3-5x faster inference locally.

In [ ]:
# Export each model to ONNX
# The trainer saves checkpoints to experiments/models/{model_name}_best.pth
# We export to models/{model_name}_retinopathy.onnx (what model_manager expects)

models = ["swin", "maxvit", "convnext_v2", "efficientnet_v2", "deit"]

for model in models:
    checkpoint = f"experiments/models/{model}_best.pth"
    output = f"models/{model}_retinopathy.onnx"
    print(f"\nExporting {model}...")
    !python export_onnx.py --model {model} --checkpoint {checkpoint} --output {output} --verify --quantize

## 6. Verify Exported Models

In [ ]:
import os

print("Exported ONNX models:")
for f in os.listdir("models"):
    if f.endswith(".onnx"):
        size_mb = os.path.getsize(os.path.join("models", f)) / 1e6
        print(f"  {f}: {size_mb:.1f} MB")

## 7. Quick Inference Test

In [ ]:
import time

import numpy as np
import onnxruntime as ort

categories = ["Healthy", "Cataract", "Glaucoma", "Retina Disease"]

# Test each ONNX model
for model_name in models:
    onnx_path = f"models/{model_name}_retinopathy.onnx"
    if not os.path.exists(onnx_path):
        print(f"  {model_name}: NOT FOUND")
        continue

    session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

    # Create dummy input
    dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)

    # Benchmark
    start = time.time()
    for _ in range(10):
        output = session.run(None, {"input": dummy})
    elapsed = (time.time() - start) / 10

    probs = np.exp(output[0][0]) / np.exp(output[0][0]).sum()
    pred = categories[np.argmax(probs)]
    print(f"  {model_name}: {pred} ({max(probs) * 100:.1f}%) — {elapsed * 1000:.1f}ms/image")

## 8. Download Trained Models

Download the `.onnx` files to your local machine, then place them in your project's `models/` folder.

In [ ]:
# Zip and download all ONNX models
!zip -j trained_models.zip models/*.onnx

from google.colab import files

files.download("trained_models.zip")

print("\nDownload complete!")
print("Unzip and place the .onnx files into your local project's models/ folder.")

## 9. (Optional) Download PyTorch Checkpoints

If you also want the raw `.pth` checkpoints.

In [ ]:
# Zip and download PyTorch checkpoints
!zip -j trained_checkpoints.zip experiments/models/*.pth

from google.colab import files

files.download("trained_checkpoints.zip")